# importar librerias necesarias

In [1]:
# instalacion de selenium y webdriver-manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [2]:
# Configuración del navegador para Codespaces
edge_options = EdgeOptions()
edge_options.add_argument("--headless=new")
edge_options.add_argument("--no-sandbox")
edge_options.add_argument("--disable-dev-shm-usage")
edge_options.add_argument("--disable-gpu")

driver = webdriver.Edge(options=edge_options)
wait = WebDriverWait(driver, 10)

# URL del subreddit (ejemplo con top del día)
subreddit_url = "https://www.reddit.com/r/wallstreetbets/"
post_data = []
visited_links = set()

In [3]:
try:
    # 1. Cargar la página principal del subreddit
    driver.get(subreddit_url)
    time.sleep(4)  # Espera inicial para la carga dinámica
    print(f"Cargada página principal: {driver.title}")

    # 2. Encontrar todos los contenedores de artículos
    # El selector "article.m-full.m-0" busca el elemento <article> con esas dos clases.
    article_containers = driver.find_elements(By.CSS_SELECTOR, "article.m-full.m-0")
    print(f"Encontrados {len(article_containers)} contenedores 'article'.")

    # 3. Extraer enlaces únicos de los artículos
    post_links = []
    for article in article_containers:
        try:
            # Busca un elemento <a> con el atributo 'data-click-id' igual a 'body' dentro del artículo.
            link_element = article.find_element(By.CSS_SELECTOR, "a[data-click-id='body']")
            link = link_element.get_attribute("href")
            if link and link not in visited_links:
                post_links.append(link)
                visited_links.add(link)
        except Exception as e:
            print(f"  No se pudo extraer enlace de un artículo: {e}")
            continue

    print(f"Extraídos {len(post_links)} enlaces únicos de posts.")

    # 4. Visitar cada enlace y extraer la información
    for i, post_url in enumerate(post_links[:5]):  # Limita a 5 posts para prueba
        print(f"\n--- Procesando post {i+1}/{len(post_links[:5])}: {post_url[:80]}...")
        try:
            driver.get(post_url)
            # Espera a que el título principal esté presente
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

            # Extraer el título (h1)
            title_element = driver.find_element(By.TAG_NAME, "h1")
            title = title_element.text

            # Intentar extraer el contenido del contenedor específico
            # La clase que buscas: 'mb-sm mb-xs px-md xs:px-0 overflow-hidden'
            # Debemos escapar los ':' y usar puntos para las clases en CSS Selector.
            target_class = "mb-sm.mb-xs.px-md.xs\\:px-0.overflow-hidden"
            content_elements = driver.find_elements(By.CLASS_NAME, target_class)

            content_text = ""
            if content_elements:
                # Toma el texto del primer elemento encontrado con esa clase
                content_text = content_elements[0].text
            else:
                content_text = "[Contenedor no encontrado]"

            # Guardar los datos
            post_data.append({
                "title": title,
                "url": post_url,
                "content_snippet": content_text[:200] + "..." if len(content_text) > 200 else content_text  # Guarda un fragmento
            })
            print(f"  Título: {title}")
            print(f"  Contenido (inicio): {content_text[:100]}...")

            # Pausa para ser amable con el servidor
            time.sleep(2)

        except Exception as e:
            print(f"  Error procesando el post: {e}")
            continue

    # 5. Mostrar resumen y guardar en CSV
    print(f"\n{'='*50}")
    print(f"RESUMEN: Se procesaron {len(post_data)} posts exitosamente.")
    for data in post_data:
        print(f"\n- {data['title']}")

    # Guardar a CSV
    if post_data:
        import pandas as pd
        df = pd.DataFrame(post_data)
        filename = f"reddit_posts_detailed_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"\nDatos guardados en '{filename}'")

except Exception as e:
    print(f"Ocurrió un error general: {e}")
finally:
    driver.quit()
    print("\nNavegador cerrado.")

Cargada página principal: 
Encontrados 0 contenedores 'article'.
Extraídos 0 enlaces únicos de posts.

RESUMEN: Se procesaron 0 posts exitosamente.

Navegador cerrado.
